# Data Preprocessing

In [ ]:
# Remember: library imports are ALWAYS at the top of the script, no exceptions!
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from math import ceil

sns.set()

In [ ]:
## Un-comment these if you want to use ydata_profiling

# !pip install -U ydata-profiling
# from ydata_profiling import ProfileReport


## Context
The data we will be using through the pratical classes comes from a small relational database whose schema can be seen below:
![Schema](https://raw.githubusercontent.com/fpontejos/DM1_2324/main/figures/schema.png "Relation database schema")

## Reading the Data

In [ ]:
## Allow Colab to see Google Drive files

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Load csv file into a dataframe
## Paste the path here
data_path = "/content/drive/MyDrive/Colab Data/datamining.csv"

df = pd.read_csv(data_path)

In [ ]:
## Load csv file into a dataframe
## Alternative location of the csv file

## data_path = "https://raw.githubusercontent.com/fpontejos/DM1_2324/main/data/datamining.csv"

## df = pd.read_csv(data_path)

### Make a copy of your original dataset

why?

In [ ]:
df_original = df.copy()

## Metadata
- *id* - The unique identifier of the customer
- *age* - The year of birht of the customer
- *income* - The income of the customer
- *frq* - Frequency: number of purchases made by the customer
- *rcn* - Recency: number of days since last customer purchase
- *mnt* - Monetary: amount of € spent by the customer in purchases
- *clothes* - Number of clothes items purchased by the customer
- *kitchen* - Number of kitchen items purchased by the customer
- *small_appliances* - Number of small_appliances items purchased by the customer
- *toys* - Number of toys items purchased by the customer
- *house_keeping* - Number of house_keeping items purchased by the customer
- *dependents* - Binary. Whether or not the customer has dependents
- *per_net_purchase* - Percentage of purchases made online
- *education* - Education level of the customer
- *status* - Marital status of the customer
- *gender* - Gender of the customer
- *description* - Last customer's recommendation description

## Problems:
- Duplicates?
- Data types?
- Missing values?
- Strange values?
- Descriptive statistics?

### Take a closer look and point out possible problems:

(hint: a missing values in pandas is represented with a NaN value)

In [ ]:
df.dtypes

In [ ]:
# Check descriptive statistics
df.describe(include="all").T

In [ ]:
df.head()

In [ ]:
# Sometimes it is not obvious that a value is missing
# For example if the value is an empty string

# replace "" by nans
df.replace("", np.nan, inplace=True)

In [ ]:
df["dependents"] = df["dependents"].astype("boolean")

In [ ]:
# check dataset data types again
df.dtypes

In [ ]:
# check descriptive statistics again
df.describe(include="all").T

In [ ]:
# Define metric and non-metric features. Why?
non_metric_features = ["education", "status", "gender", "dependents", "description"]

## This is saying that the metric features are all the other features that are not non-metric
## Need to be careful in case not all columns are to be used as features

# metric_features = df.columns.drop(non_metric_features).to_list()

## Or you can also specify manually
metric_features = ['age',
 'income',
 'frq',
 'rcn',
 'mnt',
 'clothes',
 'kitchen',
 'small_appliances',
 'toys',
 'house_keeping',
 'per_net_purchase']

### Fill missing values (Data imputation)

How can we fill missing values?


#### Using measures of central tendency

In [ ]:
# Creating a copy to apply central tendency measures imputation
df_central = df.copy()

In [ ]:
# count of missing values
df_central.isna().sum()

In [ ]:
medians = df_central[metric_features].median()
medians

In [ ]:
modes = df_central[non_metric_features].mode().loc[0]
modes

In [ ]:
## Fill NaNs using medians and modes

df_central.fillna(medians, inplace=True)
df_central.fillna(modes, inplace=True)

In [ ]:
df_central.isna().sum()  # checking how many NaNs we still have

#### Using Nearest Neighbor imputation

In [ ]:
# Creating new df copy to explore neighbordhood imputation
df_neighbors = df.copy()

In [ ]:
# Seeing rows with NaNs
nans_index = df_neighbors.isna().any(axis=1)
df_neighbors[nans_index]


In [ ]:
# Use KNNImputer - only works for numerical variables
imputer = KNNImputer(n_neighbors=5, weights="uniform")
df_neighbors[metric_features] = imputer.fit_transform(df_neighbors[metric_features])

In [ ]:
# See rows with NaNs imputed
df_neighbors.loc[nans_index, metric_features]

In [ ]:
# let's keep the central imputation
df = df_central.copy()

### Outlier removal

Why do we need to remove outliers? Which methods can we use?


Let's start by "manually" filtering the dataset's outliers

In [ ]:
def remove_outliers(df, filters):

  df_2 = df[filters]
  print('Percentage of data kept after removing outliers:', 100*(np.round(df_2.shape[0] / df.shape[0], 4)))

  return df_2



In [ ]:
# This may vary from session to session, and is prone to varying interpretations.
# A simple example is provided below:

manual_filters = (
    (df['house_keeping']<=50)
    &
    (df['kitchen']<=40)
    &
    (df['toys']<=35)
    &
    (df['education']!='OldSchool')
)

df_1 = remove_outliers(df, manual_filters)

#### Outlier removal using only the IQR method

Why should you use/not use this method?

In [ ]:
## Remember our boxplots from previous practical session?

def plot_multiple_boxplots(data, feats, title="Numeric Variables' Box Plots", figsize=(18, 8)):

    # Prepare figure. Create individual axes where each histogram will be placed
    fig, axes = plt.subplots(2, ceil(len(feats) / 2), figsize=figsize)

    # Plot data
    # Iterate across axes objects and associate each histogram (hint: use the ax.hist() instead of plt.hist()):
    for ax, feat in zip(axes.flatten(), feats): # Notice the zip() function and flatten() method
      sns.boxplot(x=data[feat], ax=ax)
      ax.set_title(feat)
      ax.set_xlabel("")

    # Layout
    # Add a centered title to the figure:
    plt.suptitle(title)

    plt.show()

    return



In [ ]:
# All Numeric Variables' Histograms in one figure
sns.set()

plot_multiple_boxplots(df, metric_features)


In [ ]:
def get_iqr_filters(df, feats, lower_q=.25, upper_q=.75):
  q25 = df[feats].quantile(lower_q)
  q75 = df[feats].quantile(upper_q)
  iqr = (q75 - q25)

  upper_lim = q75 + 1.5 * iqr
  lower_lim = q25 - 1.5 * iqr

  iqr_filters = []
  for metric in feats:
      llim = lower_lim[metric]
      ulim = upper_lim[metric]
      iqr_filters.append(df[metric].between(llim, ulim, inclusive='both'))

  filters = pd.Series(np.all(iqr_filters, 0))


  return filters



In [ ]:
iqr_filters = get_iqr_filters(df, metric_features)

df_iqr = remove_outliers(df, iqr_filters)

What do you think about this percentage?

### Combining different outlier methods

More robust/ consistent outlier detection method:

In [ ]:
df_3 = df[(manual_filters | iqr_filters)]
print('Percentage of data kept after removing outliers:', 100*(np.round(df_3.shape[0] / df_original.shape[0], 4)))

In [ ]:
# Get the manual filtering version
df = df_1.copy()

#### Remember the "rcn" histogram?

In [ ]:
df['rcn'].hist()

In [ ]:
# How can we avoid having as many extreme values in 'rcn'?
print((df['rcn']>100).value_counts())

rcn_t = df['rcn'].copy()
rcn_t.loc[rcn_t>100] = 100

df['rcn'] = rcn_t

#### What about non-metric features?

![](https://raw.githubusercontent.com/fpontejos/DM1_2324/main/figures/exp_analysis/categorical_variables_frequecies.png)

In [ ]:
# Let's also remove status=Whatever
df.loc[df['status'] == 'Whatever', 'status'] = df['status'].mode()[0]


### Feature Engineering and Feature Selection

A reminder of our metadata:
- *id* - The unique identifier of the customer
- *age* - The year of birht of the customer
- *income* - The income of the customer
- *frq* - Frequency: number of purchases made by the customer
- *rcn* - Recency: number of days since last customer purchase
- *mnt* - Monetary: amount of € spent by the customer in purchases
- *clothes* - Number of clothes items purchased by the customer
- *kitchen* - Number of kitchen items purchased by the customer
- *small_appliances* - Number of small_appliances items purchased by the customer
- *toys* - Number of toys items purchased by the customer
- *house_keeping* - Number of house_keeping items purchased by the customer
- *dependents* - Binary. Whether or not the customer has dependents
- *per_net_purchase* - Percentage of purchases made online
- *education* - Education level of the customer
- *status* - Marital status of the customer
- *gender* - Gender of the customer
- *description* - Last customer's recommendation description

In [ ]:
df['birth_year'] = df['age']
df['age'] = datetime.now().year - df['birth_year']

df['spent_online'] = (df['per_net_purchase'] / 100) * df['mnt']

### Variable selection: Redundancy VS Relevancy

### Redundancy
We already saw our original correlation matrix:
![Correlation Matrix](https://raw.githubusercontent.com/fpontejos/DM1_2324/main/figures/exp_analysis/correlation_matrix.png)

In [ ]:
# Select variables according to their correlations
df.drop(columns=['birth_year', 'age', 'mnt'], inplace=True)

In [ ]:
## Reminder of our metric features
metric_features

In [ ]:
# Updating metric_features
metric_features.append("spent_online")
metric_features.remove("mnt")
metric_features.remove("age")

In [ ]:
metric_features

### Relevancy
Selecting variables based on the relevancy of each one to the task. Example: remove uncorrelated variables with the target, stepwise regression, use variables for product clustering, use variables for socio-demographic clustering, ...

Variables that aren't correlated with any other variable are often also not relevant. In this case we will not focus on this a lot since we don't have a defined task yet.

### Data Normalization

#### MinMax Scaling

In [ ]:
df_minmax = df.copy()

In [ ]:
df_minmax.shape

In [ ]:
# Use MinMaxScaler to scale the data
scaler = MinMaxScaler()
scaled_feat = scaler.fit_transform(df_minmax[metric_features])
scaled_feat

In [ ]:
df_minmax[metric_features] = scaled_feat
df_minmax.head()

In [ ]:
# Checking max and min of minmaxed variables
df_minmax[metric_features].describe().round(2)

#### Standard Scaling

In [ ]:
df_standard = df.copy()

In [ ]:
scaler = StandardScaler()
scaled_feat = scaler.fit_transform(df_standard[metric_features])
scaled_feat

In [ ]:
df_standard[metric_features] = scaled_feat
df_standard.head()

In [ ]:
# Checking mean and variance of standardized variables
df_standard[metric_features].describe().round(2)

In [ ]:
df = df_standard.copy()

### One-hot encoding

Why do we need to do this?

In [ ]:
df_ohc = df.copy()

In [ ]:
def get_ohc_df(df, feats):
  # Use OneHotEncoder to encode the categorical features.
  # Get feature names and create a DataFrame
  # with the one-hot encoded categorical features (pass feature names)

  ohc = OneHotEncoder(sparse_output=False, drop="first")
  ohc_feat = ohc.fit_transform(df[feats])
  ohc_feat_names = ohc.get_feature_names_out()
  ohc_df = pd.DataFrame(ohc_feat, index=df.index, columns=ohc_feat_names)

  # Reassigning df to contain ohc variables
  df_ohc = pd.concat([df, ohc_df], axis=1)

  return df_ohc, ohc

df_ohc, ohc = get_ohc_df(df, non_metric_features)


In [ ]:
df_ohc.shape

In [ ]:
ohc_feats = ohc.get_feature_names_out().tolist()
ohc_feats

In [ ]:
df_ohc.columns

In [ ]:
df = df_ohc.copy()

## Next: Clustering Algorithms

### Questions?

## [OPTIONAL] Exercise

To practice your data preprocessing skills, you can do the same exercises on a different dataset.


The Spaceship Titanic Dataset has been loaded for you in the cells below. You can find more information about this dataset from the Kaggle link.


Using this notebook as a guide, try to answer the questions that follow.


---

Addison Howard, Ashley Chow, Ryan Holbrook. (2022). Spaceship Titanic. Kaggle. https://kaggle.com/competitions/spaceship-titanic


In [ ]:
titanic_df = pd.read_csv("https://raw.githubusercontent.com/fpontejos/DM1_2324/main/data/spaceship_titanic_dataset.csv")


In [ ]:
## Are all the features useful? It might be helpful to read the dataset description on Kaggle.
## Keep only the useful features

titanic_metric_features = []
titanic_non_metric_features = []


In [ ]:
## Do you have any missing values?


In [ ]:
## Replace missing values using one of the methods shown previously


In [ ]:
## Do you have any outliers?


In [ ]:
## Would it make sense to remove the outliers using the IQR method?
## Why / Why not?


In [ ]:
## Handle the outliers


In [ ]:
## Are any of the features redundant?


In [ ]:
## Are any of the features not relevant?


In [ ]:
## Do you need to perform scaling?


In [ ]:
## Do you need to transform the categorical features?


In [ ]:
## Have you fixed all the issues you identified in your dataset?
